# EvoBrain: Multi-Horizon Future Prediction & Seizure Classification on GPU

This notebook trains **EvoBrain** and its baselines with a **Dual-Head Multi-Task Objective**:
1. **Head 1 (Time Proximity)**: Predicts **when** a seizure occurs across 5 horizons:
   - `now` : Seizure active right now
   - `5s`  : Onset within next 5 seconds (Imminent)
   - `10s` : Onset within next 10 seconds (Warning)
   - `20s` : Onset within next 20 seconds (Alert)
   - `40s` : Onset within next 40 seconds (Advisory)
   - **Delta t > 40s**: Strictly set to **0** (`[0, 0, 0, 0, 0]` = Safe Background)
2. **Head 2 (Seizure Subtype)**: Predicts **what kind** of seizure it is / will be:
   - `0: bckg` (Normal baseline brainwaves)
   - `1: fnsz` (Focal Non-Specific Seizure)
   - `2: gnsz` (Generalized Non-Specific Seizure)
   - `3: cpsz` (Complex Partial Seizure)
   - `4: other` (Tonic-Clonic, Absence, Myoclonic, etc.)

### Models Supported
- `lstm_baseline`: Fast sequential temporal baseline.
- `gru_gcn_baseline`: Spatial-temporal GCN with GRU (**FPGA target**).
- `evobrain_gru`: EvoBrain with GRU backbone.
- `og_evobrain`: Original NeurIPS 2025 Spotlight (Two-Stream Mamba + Laplacian PE + SSGConv).

In [ ]:
# @title 1. GPU Check & Environment Setup
import torch
import sys
import os

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

# Install dependencies
!pip install -q torch-geometric datasets huggingface_hub pyedflib scikit-learn more-itertools h5py

# Install Mamba SSM & Causal Conv1D for the OG EvoBrain
try:
    !pip install -q causal-conv1d mamba-ssm
    print("mamba-ssm installed successfully!")
except Exception as e:
    print(f"Notice: mamba-ssm install failed ({e}). You can still run LSTM, GRU-GCN, and EvoBrain-GRU.")

In [ ]:
# @title 2. Mount Google Drive & Clone Custom Fork (nruaif/EvoBrain)
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/EvoBrain'):
    !git clone https://github.com/nruaif/EvoBrain.git /content/EvoBrain
else:
    !git -C /content/EvoBrain remote set-url origin https://github.com/nruaif/EvoBrain.git
    !git -C /content/EvoBrain pull origin main

sys.path.append('/content/EvoBrain')
os.chdir('/content/EvoBrain')
print("Working directory set to /content/EvoBrain (Fork: nruaif/EvoBrain)")

In [ ]:
# @title 3. Dataset: Soft Time Horizons [now, 5s, 10s, 20s, 40s] + Seizure Class
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from scipy.fftpack import fft
import numpy as np
import json

HF_REPO_ID = "your-username/tusz-evobrain-19ch-200hz"  # @param {type:"string"}
BATCH_SIZE = 128                                       # @param {type:"integer"}
NUM_WORKERS = 2                                        # @param {type:"integer"}
TOP_K = 3                                              # Sparsity for dynamic graph

# Seizure Subtype Map
CLASS_MAP = {"bckg": 0, "fnsz": 1, "gnsz": 2, "cpsz": 3}

try:
    hf_train = load_dataset(HF_REPO_ID, split="train")
    hf_dev = load_dataset(HF_REPO_ID, split="dev")
    print(f"Loaded HF train: {len(hf_train)} recordings | dev: {len(hf_dev)} recordings")
except Exception as e:
    print(f"Could not load from Hugging Face: {e}")

def compute_step_fft(signal_1s, n_fft=200):
    fourier = fft(signal_1s, n=n_fft, axis=-1)
    amp = np.abs(fourier[:, :int(n_fft // 2)])  # 100 positive frequency bins
    amp[amp == 0.0] = 1e-8
    return np.log(amp)

def compute_step_adjacency(step_raw, top_k=TOP_K):
    norms = np.linalg.norm(step_raw, axis=-1, keepdims=True)
    norms[norms == 0.0] = 1e-8
    normed = step_raw / norms
    corr = np.abs(normed @ normed.T)
    np.fill_diagonal(corr, 1.0)

    corr_no_self = corr.copy()
    np.fill_diagonal(corr_no_self, 0.0)
    topk_idx = (-corr_no_self).argsort(axis=-1)[:, :top_k]
    mask = np.eye(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        mask[i, topk_idx[i]] = True
    return (mask * corr).astype(np.float32)

def compute_time_and_class_targets(w_start, w_end, events_bi, events_multi):
    """
    Generates 5-horizon soft time vector: [now, 5s, 10s, 20s, 40s].
    When delta_t > 40s, all time targets are strictly 0.0!
    """
    time_target = np.zeros(5, dtype=np.float32)  # [now, 5s, 10s, 20s, 40s]
    class_target = 0                             # 0 = bckg

    for ev in events_bi:
        s_on, s_off = ev["start"], ev["stop"]

        # 1. Active seizure occurring right now
        if max(w_start, s_on) < min(w_end, s_off):
            time_target[0] = 1.0  # now
            time_target[1:] = 1.0
            break

        # 2. Upcoming seizure in the future
        elif s_on > w_end:
            delta_t = s_on - w_end
            if delta_t <= 40.0:
                # Exponential soft labels for smooth transition
                time_target[4] = float(np.exp(-delta_t / 40.0))  # 40s
                if delta_t <= 20.0:
                    time_target[3] = float(np.exp(-delta_t / 20.0))  # 20s
                if delta_t <= 10.0:
                    time_target[2] = float(np.exp(-delta_t / 10.0))  # 10s
                if delta_t <= 5.0:
                    time_target[1] = float(np.exp(-delta_t / 5.0))   # 5s
                break
            # When delta_t > 40s, time_target remains [0, 0, 0, 0, 0]!

    # Determine seizure subtype if active or within warning window
    if time_target.max() > 0.25:
        for ev in events_multi:
            subtype = ev.get("label", "other")
            class_target = CLASS_MAP.get(subtype, 4)  # 4 = other
            break

    return time_target, class_target

class EvoBrainMultiTaskDataset(Dataset):
    def __init__(self, hf_ds, clip_len_sec=12, stride_sec=4, is_train=True):
        self.hf_ds = hf_ds
        self.clip_len_sec = clip_len_sec
        self.stride_sec = stride_sec
        self.samples = []

        ds_meta = hf_ds.remove_columns(["eeg_fp16_bytes"])
        for row_idx in range(len(ds_meta)):
            row = ds_meta[row_idx]
            sfreq = row["sfreq"]
            n_samples = row["n_samples"]
            win_samples = int(clip_len_sec * sfreq)
            stride_samples = int(stride_sec * sfreq)
            if n_samples < win_samples:
                continue

            events_bi = json.loads(row["events_bi_json"])
            events_multi = json.loads(row["events_multi_json"])

            for start_s in range(0, n_samples - win_samples + 1, stride_samples):
                end_s = start_s + win_samples
                w_start = start_s / sfreq
                w_end = end_s / sfreq

                t_target, c_target = compute_time_and_class_targets(w_start, w_end, events_bi, events_multi)
                self.samples.append((row_idx, start_s, end_s, t_target, c_target))

        # Class balancing for training set (balance positive warning/seizure vs pure background)
        if is_train:
            pos = [s for s in self.samples if s[3].max() > 0.0]
            neg = [s for s in self.samples if s[3].max() == 0.0]
            np.random.shuffle(neg)
            self.samples = pos + neg[:len(pos)]
            np.random.shuffle(self.samples)
            print(f"Train Balanced: {len(pos)} Warning/Seizure clips + {len(pos)} Background clips = {len(self.samples)} Total")
        else:
            print(f"Dev/Val Set  : {len(self.samples)} Total clips")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row_idx, start_s, end_s, t_target, c_target = self.samples[idx]
        row = self.hf_ds[row_idx]
        sfreq = row["sfreq"]
        eeg = np.frombuffer(row["eeg_fp16_bytes"], dtype=np.float16).reshape(19, row["n_samples"])
        clip = eeg[:, start_s:end_s].astype(np.float32)

        fft_steps = []
        adj_steps = []
        step_samples = int(sfreq)
        for t in range(self.clip_len_sec):
            step_raw = clip[:, t*step_samples : (t+1)*step_samples]
            fft_steps.append(compute_step_fft(step_raw, n_fft=step_samples))
            adj_steps.append(compute_step_adjacency(step_raw, top_k=TOP_K))

        x_fft = np.stack(fft_steps, axis=0)   # (12, 19, 100)
        adj_seq = np.stack(adj_steps, axis=0) # (12, 19, 19)

        return {
            "x": torch.from_numpy(x_fft).float(),
            "adj": torch.from_numpy(adj_seq).float(),
            "time_target": torch.from_numpy(t_target).float(),  # (5,) [now, 5s, 10s, 20s, 40s]
            "class_target": torch.tensor(c_target).long(),       # (scalar) 0..4
            "seq_len": torch.tensor(self.clip_len_sec).long()
        }

train_ds = EvoBrainMultiTaskDataset(hf_train, clip_len_sec=12, stride_sec=3, is_train=True)
dev_ds = EvoBrainMultiTaskDataset(hf_dev, clip_len_sec=12, stride_sec=6, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("Multi-Horizon DataLoaders ready!")

In [ ]:
# @title 4. Multi-Task Model (Choi et al. CoRN-LN + Qian et al. Zero-DSP LSTM)
# Implements:
# 1. "Hardware-Efficient Softmax and Layer Normalization with Guaranteed Normalization for Edge Devices" (Choi et al., arXiv:2604.23647, Apr 2026)
# 2. "Energy Efficient LSTM Accelerators for Embedded FPGAs through Parameterised Architecture Design" (Qian et al., arXiv:2604.19293, Apr 2026)
# 3. "Micro-Batch Training with Batch-Channel Normalization and Weight Standardization" (Qiao et al., TPAMI 2020)
import torch.nn as nn
from types import SimpleNamespace
from torch.nn.utils import parametrize

MODEL_CHOICE = "lstm_baseline"      # @param ["lstm_baseline", "gru_gcn_baseline", "evobrain_gru", "og_evobrain"]
MODEL_SCALE = "large_5.5M"         # @param ["small_108K", "medium_1.5M", "large_5.5M", "xlarge_18M"]
NORM_TYPE = "corn_ln"              # @param ["corn_ln", "bcn_ws", "gn_ws", "bn"]
LSTM_ACT_MODE = "hard"             # @param ["hard", "standard"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SCALE_PRESETS = {
    "small_108K":  {"rnn_units": 64,  "embed_dim": 64,  "mlp_dim": 128, "num_rnn_layers": 1, "skip": False, "agg": "max"},
    "medium_1.5M": {"rnn_units": 128, "embed_dim": 128, "mlp_dim": 256, "num_rnn_layers": 2, "skip": True,  "agg": "concat"},
    "large_5.5M":  {"rnn_units": 256, "embed_dim": 256, "mlp_dim": 512, "num_rnn_layers": 2, "skip": True,  "agg": "concat"},
    "xlarge_18M":  {"rnn_units": 512, "embed_dim": 512, "mlp_dim": 1024,"num_rnn_layers": 3, "skip": True,  "agg": "concat"},
}

cfg = SCALE_PRESETS[MODEL_SCALE]
args = SimpleNamespace(
    num_nodes=19,
    input_dim=100,
    embed_dim=cfg["embed_dim"],
    rnn_units=cfg["rnn_units"],
    num_rnn_layers=cfg["num_rnn_layers"],
    num_classes=64,
    dropout=0.2,
    max_seq_len=12,
    agg=cfg["agg"],
    mlp_dim=cfg["mlp_dim"],
    skip=cfg["skip"],
    norm_type=NORM_TYPE,
    use_corn_ln=True,
    act_mode=LSTM_ACT_MODE,
)

# Qian et al. (2026): Zero-DSP Activation Functions for Embedded FPGAs
class HardSigmoidStar(nn.Module):
    """Slope 0.125 = 1/8 = 2^(-3). On FPGA: (x >> 3) + 0.5. Consumes 3 LUTs, 0 DSPs."""
    def __init__(self, slope=0.125, offset=0.5):
        super().__init__()
        self.slope = slope
        self.offset = offset
    def forward(self, x):
        return torch.clamp(x * self.slope + self.offset, 0.0, 1.0)

class HardTanh_FPGA(nn.Module):
    """Piecewise linear Tanh approximation clamp([-1, 1]). Consumes 5 LUTs, 0 DSPs."""
    def __init__(self, min_val=-1.0, max_val=1.0):
        super().__init__()
        self.min_val = min_val
        self.max_val = max_val
    def forward(self, x):
        return torch.clamp(x, self.min_val, self.max_val)

# Choi et al. (2026): CoRN-LN LayerNorm & Mul-/Div-Free Softmax
class CoRNLayerNorm(nn.Module):
    """Compressed Reciprocal Newton Layer Normalization (14x smaller, 1199 um^2, sigma=1.0)."""
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.weight = nn.Parameter(torch.ones(self.normalized_shape))
            self.bias = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)

    def forward(self, x):
        dims = tuple(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = torch.mean((x - mean) ** 2, dim=dims, keepdim=True) + self.eps
        lod_k = torch.floor(torch.log2(var))
        x0 = torch.pow(2.0, -0.5 * lod_k)
        x1 = 0.5 * (x0 + 1.0 / (x0 * var))
        x2 = 0.5 * (x1 + 1.0 / (x1 * var))
        out = (x - mean) * x2
        if self.elementwise_affine:
            out = out * self.weight + self.bias
        return out

class MulDivFreeSoftmax(nn.Module):
    """Divider-free Softmax via Radix R=8 2-LUT exp + shift-subtract divider (11x smaller, 942 um^2, sum(p)=1.0)."""
    def __init__(self, dim=-1, R=8):
        super().__init__()
        self.dim = dim
        self.R = R

    def forward(self, x):
        max_val = torch.max(x, dim=self.dim, keepdim=True)[0]
        delta = max_val - x
        frac = torch.floor(delta / self.R)
        rem = delta - self.R * frac
        frac_clamped = torch.clamp(frac, 0, 7)
        coarse = torch.exp(-self.R * frac_clamped)
        coarse = torch.where(frac > 7, torch.zeros_like(coarse), coarse)
        residual = torch.exp(-rem)
        y = coarse * residual
        z = torch.sum(y, dim=self.dim, keepdim=True)
        p = y / (z + 1e-12)
        p = p / p.sum(dim=self.dim, keepdim=True)
        return p

class WeightStandardizer(nn.Module):
    """Weight Standardization (WS, Qiao et al. TPAMI 2020). Smooths loss landscape by reducing Lipschitz constants."""
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps
    def forward(self, w):
        dim = -1 if w.ndim >= 2 else 0
        mean = w.mean(dim=dim, keepdim=True)
        var = torch.var(w, dim=dim, keepdim=True, unbiased=False)
        return (w - mean) / torch.sqrt(var + self.eps)

class BCN1d(nn.Module):
    """Batch-Channel Normalization (BCN, Qiao et al.): BCN(X) = GN(BN(X))."""
    def __init__(self, channels, groups=None, eps=1e-5):
        super().__init__()
        if groups is None:
            for g in [16, 8, 4, 2, 1]:
                if channels % g == 0:
                    groups = g
                    break
        self.bn = nn.BatchNorm1d(channels, eps=eps)
        self.gn = nn.GroupNorm(num_groups=groups, num_channels=channels, eps=eps)
    def forward(self, x):
        return self.gn(self.bn(x))

def make_norm(channels, norm_type=NORM_TYPE, eps=1e-5):
    if norm_type in ["corn_ln", "corn_ws"]:
        return CoRNLayerNorm(channels, eps=eps)
    elif norm_type == "bcn_ws":
        return BCN1d(channels, eps=eps)
    elif norm_type == "gn_ws":
        groups = 1
        for g in [16, 8, 4, 2, 1]:
            if channels % g == 0:
                groups = g
                break
        return nn.GroupNorm(num_groups=groups, num_channels=channels, eps=eps)
    return nn.BatchNorm1d(channels, eps=eps)

def make_head(in_dim, mlp_dim, out_dim, norm_type=NORM_TYPE, dropout=0.2):
    """Hardware-optimized head using CoRN-LN / BCN + ReLU."""
    if mlp_dim <= 0:
        return nn.Linear(in_dim, out_dim)
    return nn.Sequential(
        nn.Linear(in_dim, mlp_dim, bias=False),
        make_norm(mlp_dim, norm_type),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(mlp_dim, mlp_dim // 2, bias=False),
        make_norm(mlp_dim // 2, norm_type),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(mlp_dim // 2, out_dim),
    )

class MultiTaskSeizureModel(nn.Module):
    def __init__(self, backbone_model, hidden_dim=None, mlp_dim=512, norm_type=NORM_TYPE, num_classes=5):
        super().__init__()
        self.backbone = backbone_model
        if hidden_dim is None:
            hidden_dim = backbone_model.fc.in_features
        self.hidden_dim = hidden_dim

        self.backbone_norm = make_norm(hidden_dim, norm_type)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.2)

        # Head 1: Time Proximity across 5 horizons [now, 5s, 10s, 20s, 40s]
        self.time_head = make_head(hidden_dim, mlp_dim, 5, norm_type=norm_type)
        # Head 2: Seizure Subtype [bckg, fnsz, gnsz, cpsz, other]
        self.class_head = make_head(hidden_dim, mlp_dim, num_classes, norm_type=norm_type)
        # Hardware-efficient Mul-/Div-free Softmax (Choi et al. 2026)
        self.softmax = MulDivFreeSoftmax(dim=-1)

    def forward(self, x, seq_lens, adj=None):
        if adj is not None:
            _, hidden = self.backbone(x, seq_lens, adj)
        else:
            _, hidden = self.backbone(x, seq_lens)

        if hidden.dim() == 3:
            hidden = torch.max(hidden, dim=1)[0]  # (B, hidden_dim)

        feat = self.dropout(self.relu(self.backbone_norm(hidden)))
        time_logits = self.time_head(feat)    # (B, 5)
        class_logits = self.class_head(feat)  # (B, 5)
        return time_logits, class_logits

    def predict_probabilities(self, time_logits, class_logits):
        """Hardware-efficient probability prediction with guaranteed normalization."""
        return self.softmax(time_logits), self.softmax(class_logits)

# Instantiate chosen backbone
if MODEL_CHOICE == "lstm_baseline":
    from model.lstm import LSTMModel
    base_model = LSTMModel(args, num_classes=64, device=device, use_corn_ln=True, act_mode=LSTM_ACT_MODE)
elif MODEL_CHOICE == "gru_gcn_baseline":
    from model.gru_gcn import GRU_GCN_classification, GRU_GCN
    base_model = GRU_GCN_classification(args=args, num_classes=64, device=device)
    base_model.gru_gcn = GRU_GCN(
        feat_input_size_edge=args.input_dim,
        feat_input_size_node=args.input_dim,
        feat_target_size=args.rnn_units,
        embed_inside_size=args.embed_dim,
        convolve="gcn",
        reduce_edge="gru",
        reduce_node="gru",
        skip=args.skip,
        activate="tanh",
        concat=True,
        neo_gnn=True,
    )
    base_model.gru_gcn.activate = nn.ReLU()
    base_model.gru_gcn.gnnx2.activate = nn.ReLU()
    base_model.gru_gcn.gnnx2.edge_activate = nn.ReLU()
    if args.num_rnn_layers > 1:
        base_model.gru_gcn.snn_node = nn.GRU(args.input_dim, args.embed_dim, num_layers=args.num_rnn_layers, dropout=args.dropout)
        base_model.gru_gcn.snn_edge = nn.GRU(1, args.embed_dim, num_layers=args.num_rnn_layers, dropout=args.dropout)
    fc_in = base_model.gru_gcn.feat_target_size * (args.num_nodes if args.agg == "concat" else 1)
    base_model.fc = nn.Linear(fc_in, 64)
elif MODEL_CHOICE == "evobrain_gru":
    from model.EvoBrain import EvoBrain_classification
    base_model = EvoBrain_classification(args=args, num_classes=64, device=device, gnn="ssg", num_eigenvectors=8)
    base_model.evobrain.reduce_node = "gru"
    base_model.evobrain.reduce_edge = "gru"
    base_model.evobrain.activate = nn.ReLU()
    base_model.evobrain.edge_activate = nn.ReLU()
    base_model.evobrain.gnnx2.activate = nn.ReLU()
    base_model.evobrain.snn_node = nn.GRU(args.input_dim, args.input_dim, num_layers=args.num_rnn_layers, dropout=args.dropout if args.num_rnn_layers > 1 else 0.0).to(device)
    base_model.evobrain.snn_edge = nn.GRU(1, args.input_dim, num_layers=args.num_rnn_layers, dropout=args.dropout if args.num_rnn_layers > 1 else 0.0).to(device)
    fc_in = (base_model.evobrain.feat_target_size + 8) * (args.num_nodes if args.agg == "concat" else 1)
    base_model.fc = nn.Linear(fc_in, 64)
elif MODEL_CHOICE == "og_evobrain":
    from model.EvoBrain import EvoBrain_classification
    base_model = EvoBrain_classification(args=args, num_classes=64, device=device, gnn="ssg", num_eigenvectors=8)
    base_model.evobrain.activate = nn.ReLU()
    base_model.evobrain.edge_activate = nn.ReLU()
    base_model.evobrain.gnnx2.activate = nn.ReLU()
    fc_in = (base_model.evobrain.feat_target_size + 8) * (args.num_nodes if args.agg == "concat" else 1)
    base_model.fc = nn.Linear(fc_in, 64)

backbone_dim = base_model.fc.in_features
model = MultiTaskSeizureModel(base_model, hidden_dim=backbone_dim, mlp_dim=args.mlp_dim, norm_type=NORM_TYPE, num_classes=5).to(device)

# Apply Weight Standardization (WS) if requested
if "ws" in NORM_TYPE:
    ws_count = 0
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv1d, nn.Conv2d)):
            if not parametrize.is_parametrized(m, "weight"):
                parametrize.register_parametrization(m, "weight", WeightStandardizer())
                ws_count += 1
    print(f"Weight Standardization (WS) active on {ws_count} Linear/Conv layers!")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Initialized: [{MODEL_CHOICE}] | Scale: [{MODEL_SCALE}] | Norm: [{NORM_TYPE.upper()}] | Act: [{LSTM_ACT_MODE.upper()}] | Dim: {backbone_dim}")
print(f"Total Trainable Parameters: {total_params:,} (~{total_params * 4 / (1024**2):.2f} MB in FP32)")


In [ ]:
# @title 5. Multi-Task Training Loop (Muon + AdamW / Time BCE + Class CrossEntropy)
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

OPTIMIZER_CHOICE = "muon"  # @param ["muon", "adamw"]
NUM_EPOCHS = 30
MUON_LR = 0.02             # Muon LR for 2D hidden matrices (Newton-Schulz whitened)
ADAMW_LR = 1e-3            # AdamW LR for 1D biases, LayerNorms, and final 5-output heads
TIME_LOSS_WEIGHT = 1.0
CLASS_LOSS_WEIGHT = 0.5
CHECKPOINT_DIR = "/content/drive/MyDrive/evobrain_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

@torch.compile
def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Quintic Newton-Schulz iteration to orthogonalize/whiten 2D gradient matrices."""
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X /= (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, weight_decay=0.0, normalize_weights=False):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, weight_decay=weight_decay, normalize_weights=normalize_weights)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            wd = group["weight_decay"]
            norm_w = group["normalize_weights"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.mul_(momentum).add_(g)
                g = g.add(buf, alpha=momentum) if group["nesterov"] else buf

                if norm_w:
                    p.data.mul_(len(p.data)**0.5 / (p.data.norm() + 1e-7))
                elif wd > 0:
                    p.data.mul_(1.0 - lr * wd)

                update = zeropower_via_newtonschulz5(g.reshape(len(g), -1)).view(g.shape).to(p.dtype)
                # Scale update by aspect ratio sqrt(max(1, d_out / d_in)) for rectangular RNN/MLP matrices
                scale = max(1.0, p.size(0) / p.size(1)) ** 0.5
                p.data.add_(update, alpha=-lr * scale)

criterion_time = nn.BCEWithLogitsLoss()
criterion_class = nn.CrossEntropyLoss()

# Split parameters: Muon ONLY for internal 2D+ matrices; AdamW for 1D biases/norms & final 5-class output layers
muon_params, adamw_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    is_final_out = any(k in name for k in ["time_head.8", "class_head.8", "backbone.fc"]) or (
        args.mlp_dim == 0 and any(k in name for k in ["time_head", "class_head"])
    )
    if OPTIMIZER_CHOICE == "muon" and p.ndim >= 2 and not is_final_out:
        muon_params.append(p)
    else:
        adamw_params.append(p)

optimizers = [optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=1e-4)]
if len(muon_params) > 0:
    optimizers.append(Muon(muon_params, lr=MUON_LR, momentum=0.95, nesterov=True, weight_decay=1e-4))
print(f"Optimizer: [{OPTIMIZER_CHOICE.upper()}] | Muon 2D tensors: {len(muon_params)} | AdamW 1D/Head tensors: {len(adamw_params)}")

schedulers = [optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3) for opt in optimizers]

best_auroc = 0.0
train_losses, val_aurocs_now, val_aurocs_10s = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]")
    for batch in pbar:
        x = batch["x"].to(device)             # (B, 12, 19, 100)
        adj = batch["adj"].to(device)         # (B, 12, 19, 19)
        t_target = batch["time_target"].to(device)  # (B, 5)
        c_target = batch["class_target"].to(device) # (B,)
        seq_lens = batch["seq_len"].to(device)

        model.zero_grad(set_to_none=True)
        if MODEL_CHOICE == "lstm_baseline":
            t_logits, c_logits = model(x, seq_lens)
        else:
            t_logits, c_logits = model(x, seq_lens, adj)

        loss_time = criterion_time(t_logits, t_target)
        loss_class = criterion_class(c_logits, c_target)
        loss = (TIME_LOSS_WEIGHT * loss_time) + (CLASS_LOSS_WEIGHT * loss_class)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        for opt in optimizers:
            opt.step()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "t_loss": f"{loss_time.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- Validation Phase ---
    model.eval()
    all_t_preds, all_t_targets = [], []
    all_c_preds, all_c_targets = [], []
    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]"):
            x = batch["x"].to(device)
            adj = batch["adj"].to(device)
            t_target = batch["time_target"].to(device)
            c_target = batch["class_target"].to(device)
            seq_lens = batch["seq_len"].to(device)

            if MODEL_CHOICE == "lstm_baseline":
                t_logits, c_logits = model(x, seq_lens)
            else:
                t_logits, c_logits = model(x, seq_lens, adj)

            t_probs = torch.sigmoid(t_logits).cpu().numpy()
            c_probs = torch.softmax(c_logits, dim=-1).cpu().numpy()

            all_t_preds.extend(t_probs)
            all_t_targets.extend(t_target.cpu().numpy())
            all_c_preds.extend(c_probs.argmax(axis=-1))
            all_c_targets.extend(c_target.cpu().numpy())

    all_t_preds = np.array(all_t_preds)      # (N, 5)
    all_t_targets = np.array(all_t_targets)  # (N, 5)

    # Compute AUROC for 'now' and '10s warning'
    auroc_now = roc_auc_score((all_t_targets[:, 0] > 0.5).astype(int), all_t_preds[:, 0])
    auroc_10s = roc_auc_score((all_t_targets[:, 2] > 0.2).astype(int), all_t_preds[:, 2])
    class_acc = accuracy_score(all_c_targets, all_c_preds)

    val_aurocs_now.append(auroc_now)
    val_aurocs_10s.append(auroc_10s)
    for sched in schedulers:
        sched.step(auroc_now)

    print(f"Epoch {epoch:02d}: Loss={avg_loss:.4f} | AUROC Now={auroc_now:.4f} | AUROC 10s Future={auroc_10s:.4f} | Class Acc={class_acc:.4f}")

    if auroc_now > best_auroc:
        best_auroc = auroc_now
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_multitask_best.pt")
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "best_auroc": best_auroc}, ckpt_path)
        print(f"  ==> Saved Checkpoint to {ckpt_path}")

In [ ]:
# @title 6. Power-of-Two (PoT) Quantization-Aware Training (QAT) for Zero-DSP FPGA
# Quantizes 2D weights to {0, +/- 2^k} (PoT) or {0, +/- 2^k1 +/- 2^k2} (APoT)
# and continues training via Straight-Through Estimator (STE).

POT_MODE = "pot"       # @param ["pot", "apot"]
POT_BITS = 4           # @param {type:"integer"}
POT_EPOCHS = 12        # @param {type:"integer"}
POT_LR_SCALE = 0.25    # Fine-tuning LR multiplier relative to initial training

# Optional: Load best FP32 checkpoint before starting PoT QAT
fp32_ckpt_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_multitask_best.pt")
if os.path.exists(fp32_ckpt_path):
    ckpt = torch.load(fp32_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best FP32 model (Epoch {ckpt['epoch']}, AUROC={ckpt['best_auroc']:.4f}) for PoT QAT!")

@torch.no_grad()
def quantize_to_pot_single(w, num_bits=4):
    """Snap tensor w to nearest signed Power-of-Two: {0, +/- 2^k}."""
    abs_w = w.abs()
    max_val = abs_w.max().clamp(min=1e-8)
    # Anchor top exponent to ceil(log2(max_val)) so dynamic range is an exact power of 2
    max_exp = torch.ceil(torch.log2(max_val))
    n_levels = (1 << (num_bits - 1)) - 1  # e.g. 4-bit -> 7 exponent levels + zero
    min_exp = max_exp - n_levels + 1
    zero_thresh = 2.0 ** (min_exp - 1)

    sign = torch.sign(w)
    is_zero = abs_w < zero_thresh
    exp = torch.clamp(torch.round(torch.log2(abs_w.clamp(min=1e-12))), min=min_exp, max=max_exp)
    w_pot = torch.where(is_zero, torch.zeros_like(w), sign * (2.0 ** exp))
    return w_pot

@torch.no_grad()
def quantize_to_pot(w, num_bits=4, mode="pot"):
    """Supports single PoT (w = +/- 2^k) or Additive PoT (w = +/- 2^k1 +/- 2^k2)."""
    w1 = quantize_to_pot_single(w, num_bits=num_bits)
    if mode == "apot":
        residual = w - w1
        w2 = quantize_to_pot_single(residual, num_bits=max(3, num_bits - 1))
        return w1 + w2
    return w1

# Collect all 2D+ weight matrices to quantize to Power-of-Two
pot_target_params = [p for _, p in model.named_parameters() if p.requires_grad and p.ndim >= 2]
print(f"PoT QAT Target: {len(pot_target_params)} weight matrices | Mode: {POT_MODE.upper()} ({POT_BITS}-bit)")

# Lower learning rate for smooth QAT convergence
for opt in optimizers:
    for group in opt.param_groups:
        group["lr"] = group["lr"] * POT_LR_SCALE

best_pot_auroc = 0.0

for epoch in range(1, POT_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"PoT-QAT Epoch {epoch}/{POT_EPOCHS} [Train]")
    for batch in pbar:
        x = batch["x"].to(device)
        adj = batch["adj"].to(device)
        t_target = batch["time_target"].to(device)
        c_target = batch["class_target"].to(device)
        seq_lens = batch["seq_len"].to(device)

        model.zero_grad(set_to_none=True)

        # 1. STE Forward: Save FP32 shadow weights & snap in-place to Power-of-Two
        shadow_fp32 = [p.data.clone() for p in pot_target_params]
        for p in pot_target_params:
            p.data.copy_(quantize_to_pot(p.data, num_bits=POT_BITS, mode=POT_MODE))

        if MODEL_CHOICE == "lstm_baseline":
            t_logits, c_logits = model(x, seq_lens)
        else:
            t_logits, c_logits = model(x, seq_lens, adj)

        loss_time = criterion_time(t_logits, t_target)
        loss_class = criterion_class(c_logits, c_target)
        loss = (TIME_LOSS_WEIGHT * loss_time) + (CLASS_LOSS_WEIGHT * loss_class)

        # 2. STE Backward: Gradients are computed at exact PoT weights
        loss.backward()

        # 3. Restore FP32 master weights & apply optimizer step
        for p, w_fp32 in zip(pot_target_params, shadow_fp32):
            p.data.copy_(w_fp32)

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        for opt in optimizers:
            opt.step()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "t_loss": f"{loss_time.item():.4f}"})

    avg_loss = total_loss / len(train_loader)

    # --- Validation with 100% Snapped PoT Weights ---
    model.eval()
    eval_shadow_fp32 = [p.data.clone() for p in pot_target_params]
    total_zeros, total_elems = 0, 0
    for p in pot_target_params:
        p.data.copy_(quantize_to_pot(p.data, num_bits=POT_BITS, mode=POT_MODE))
        total_zeros += (p.data == 0).sum().item()
        total_elems += p.data.numel()
    sparsity_pct = 100.0 * total_zeros / max(1, total_elems)

    all_t_preds, all_t_targets = [], []
    all_c_preds, all_c_targets = [], []
    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"PoT-QAT Epoch {epoch}/{POT_EPOCHS} [Val]"):
            x = batch["x"].to(device)
            adj = batch["adj"].to(device)
            t_target = batch["time_target"].to(device)
            c_target = batch["class_target"].to(device)
            seq_lens = batch["seq_len"].to(device)

            if MODEL_CHOICE == "lstm_baseline":
                t_logits, c_logits = model(x, seq_lens)
            else:
                t_logits, c_logits = model(x, seq_lens, adj)

            t_probs = torch.sigmoid(t_logits).cpu().numpy()
            c_probs = torch.softmax(c_logits, dim=-1).cpu().numpy()

            all_t_preds.extend(t_probs)
            all_t_targets.extend(t_target.cpu().numpy())
            all_c_preds.extend(c_probs.argmax(axis=-1))
            all_c_targets.extend(c_target.cpu().numpy())

    all_t_preds = np.array(all_t_preds)
    all_t_targets = np.array(all_t_targets)

    auroc_now = roc_auc_score((all_t_targets[:, 0] > 0.5).astype(int), all_t_preds[:, 0])
    auroc_10s = roc_auc_score((all_t_targets[:, 2] > 0.2).astype(int), all_t_preds[:, 2])
    class_acc = accuracy_score(all_c_targets, all_c_preds)

    print(
        f"PoT Epoch {epoch:02d}: Loss={avg_loss:.4f} | PoT AUROC Now={auroc_now:.4f} | "
        f"PoT AUROC 10s={auroc_10s:.4f} | Class Acc={class_acc:.4f} | Zero-Weight Sparsity={sparsity_pct:.1f}%"
    )

    if auroc_now > best_pot_auroc:
        best_pot_auroc = auroc_now
        pot_ckpt_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_pot_qat_best.pt")
        # Save the snapped Power-of-Two weights directly!
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "best_auroc": best_pot_auroc, "pot_bits": POT_BITS, "pot_mode": POT_MODE}, pot_ckpt_path)
        print(f"  ==> Saved Best PoT-Quantized Checkpoint to {pot_ckpt_path}")

    # Restore FP32 master weights for next QAT epoch (except on final epoch)
    if epoch < POT_EPOCHS:
        for p, w_fp32 in zip(pot_target_params, eval_shadow_fp32):
            p.data.copy_(w_fp32)
    else:
        print("Finalizing model with 100% hard-snapped Power-of-Two weights!")

In [ ]:
# @title 7. Plot Multi-Horizon Prediction Trajectories
import matplotlib.pyplot as plt

horizons = ["Now (Active)", "< 5s Onset", "< 10s Onset", "< 20s Onset", "< 40s Onset"]
auroc_list = []
for h_idx in range(5):
    y_bin = (all_t_targets[:, h_idx] > 0.2).astype(int)
    if len(np.unique(y_bin)) > 1:
        score = roc_auc_score(y_bin, all_t_preds[:, h_idx])
    else:
        score = 0.5
    auroc_list.append(score)

plt.figure(figsize=(9, 4.5))
plt.bar(horizons, auroc_list, color=["#dc2626", "#ea580c", "#eab308", "#16a34a", "#2563eb"])
plt.ylim(0.5, 1.0)
plt.ylabel("AUROC Score")
plt.title(f"Multi-Horizon Early Warning Accuracy ({MODEL_CHOICE})")
for i, v in enumerate(auroc_list):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# @title 8. (Bonus) Export PoT / Dual-Head Weights & Hardware Non-GEMM C++ Header for FPGA
# Exports trained weights + HardSigmoid* + HardTanh + 2-LUT Softmax & Reciprocal Newton LayerNorm (Choi et al. + Qian et al. 2026)
pot_ckpt_file = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_pot_qat_best.pt")
fp32_ckpt_file = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_multitask_best.pt")
active_ckpt_file = pot_ckpt_file if os.path.exists(pot_ckpt_file) else fp32_ckpt_file
best_ckpt = torch.load(active_ckpt_file, map_location="cpu")
state_dict = best_ckpt["model_state_dict"]

header_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_etched_multitask_weights.h")
with open(header_path, "w") as f:
    f.write("// Auto-generated Etched Weights & Non-GEMM Hardware Units for FPGA/ASIC Acceleration\n")
    f.write(f"// Source Checkpoint: {os.path.basename(active_ckpt_file)}\n")
    f.write(f"// Architecture: {MODEL_CHOICE} (Multi-Horizon Time + Class Subtype)\n")
    f.write("// Based on:\n")
    f.write("// 1. Choi et al. (arXiv:2604.23647 / Samsung 28nm: Softmax 942 um^2, LayerNorm 1199 um^2)\n")
    f.write("// 2. Qian et al. (arXiv:2604.19293 / Spartan-7 XC7S15: 0 DSPs, 204 MHz, 11.89 GOP/s/W)\n")
    f.write("#ifndef ETCHED_WEIGHTS_H\n#define ETCHED_WEIGHTS_H\n\n")
    f.write("#include <math.h>\n#include <stdint.h>\n\n")
    
    # Qian et al. Zero-DSP activations
    f.write("// --- Qian et al. (2026): Zero-DSP Activation Functions (Fixed-Point Q4.4) ---\n")
    f.write("// HardSigmoid*: Slope 0.125 = 1/8. Arithmetic 3-bit right shift + 0.5. (Only 3 LUTs!)\n")
    f.write("inline float hardsigmoid_star(float x) {\n")
    f.write("    float val = x * 0.125f + 0.5f;\n")
    f.write("    return val < 0.0f ? 0.0f : (val > 1.0f ? 1.0f : val);\n")
    f.write("}\n\n")
    f.write("// HardTanh: Clamped to [-1.0, 1.0]. (Only 5 LUTs, 2 comparators!)\n")
    f.write("inline float hardtanh_fpga(float x) {\n")
    f.write("    return x < -1.0f ? -1.0f : (x > 1.0f ? 1.0f : x);\n")
    f.write("}\n\n")
    
    # Choi et al. Softmax Factorized LUTs (Radix R=8)
    f.write("// --- Choi et al. (2026): Algorithm 1 Mul-/Div-Free Softmax Lookup Tables (Radix R=8) ---\n")
    coarse_lut = [math.exp(-8 * i) for i in range(7)]
    f.write(f"const float SOFTMAX_LUT_COARSE[7] = {{\n    " + ", ".join(f"{v:.8e}f" for v in coarse_lut) + "\n};\n\n")
    resid_lut = [math.exp(-i) for i in range(8)]
    f.write(f"const float SOFTMAX_LUT_RESIDUAL[8] = {{\n    " + ", ".join(f"{v:.8e}f" for v in resid_lut) + "\n};\n\n")
    
    # Choi et al. Reciprocal Newton LayerNorm C++ Function
    f.write("// --- Choi et al. (2026): Algorithm 2 CoRN-LN 2-Step Reciprocal Newton rsqrt (Divider-Free) ---\n")
    f.write("inline float corn_rsqrt_2step(float var, float eps = 1e-5f) {\n")
    f.write("    float v = var + eps;\n")
    f.write("    int k = (int)floorf(log2f(v));\n")
    f.write("    float x0 = powf(2.0f, -0.5f * (float)k);\n")
    f.write("    float x1 = 0.5f * (x0 + 1.0f / (x0 * v));\n")
    f.write("    float x2 = 0.5f * (x1 + 1.0f / (x1 * v));\n")
    f.write("    return x2; // Guaranteed unit variance sigma=1.0\n")
    f.write("}\n\n")
    
    # Qian et al. 5-Stage Pipelined MAC Loop
    f.write("// --- Qian et al. (2026): 5-Stage Pipelined Fixed-Point MAC Loop ---\n")
    f.write("inline int8_t pipelined_mac_q44(const int8_t* W, const int8_t* x, int N) {\n")
    f.write("    int32_t accum = 0; // Stage 1: Init\n")
    f.write("    for (int i = 0; i < N; ++i) {\n")
    f.write("        int16_t prod = (int16_t)W[i] * (int16_t)x[i]; // Stage 3: Multiply\n")
    f.write("        accum += prod;                                 // Stage 4: Accumulate\n")
    f.write("    }\n")
    f.write("    int8_t out = (int8_t)(accum >> 4);                 // Stage 5: Round Q4.4 at end\n")
    f.write("    return out;\n")
    f.write("}\n\n")
    
    # Write Weight Arrays
    f.write("// --- Etched Model Parameters ---\n")
    for name, tensor in state_dict.items():
        sanitized_name = name.replace(".", "_")
        arr = tensor.numpy().flatten()
        f.write(f"// Parameter: {name} (Shape: {list(tensor.shape)})\n")
        f.write(f"const float {sanitized_name}[{len(arr)}] = {{\n")
        chunks = [", ".join(f"{v:.6f}f" for v in arr[i:i+8]) for i in range(0, len(arr), 8)]
        f.write("    " + ",\n    ".join(chunks) + "\n")
        f.write("};\n\n")
        
    f.write("#endif // ETCHED_WEIGHTS_H\n")

print(f"Successfully exported hardwired multi-task weights + CoRN-LN, HardSigmoid* & Softmax kernels ({os.path.basename(active_ckpt_file)}) to: {header_path}")
